In [ ]:
!pip install imbalanced-learn -q


In [ ]:
import json, os

kaggle_creds = {
    "username": "DEIVEN",
    "key": "6f9e9260622e1f076bbdd4bc30eee6dd"
}

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_creds, f)

os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("✅ kaggle.json creado correctamente")

✅ kaggle.json creado correctamente


In [ ]:
!pip install kaggle -q
!kaggle datasets download -d mrwellsdavid/unsw-nb15 --unzip -p ./data

import os
print(os.listdir('./data'))

Dataset URL: https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15
License(s): unknown
100% 149M/149M [00:01<00:00, 132MB/s]

['UNSW-NB15_4.csv', 'UNSW_NB15_training-set.csv', 'UNSW_NB15_testing-set.csv', 'UNSW-NB15_LIST_EVENTS.csv', 'NUSW-NB15_features.csv', 'UNSW-NB15_2.csv', 'UNSW-NB15_3.csv', 'UNSW-NB15_1.csv']


In [ ]:
%%writefile kalman.py
class KalmanSmoother:
    def __init__(self, sigma2_w=1e-3, sigma2_v=1e-1):
        self.Q = sigma2_w
        self.R = sigma2_v
        self.reset()

    def reset(self):
        self.s_hat = 0.5
        self.P = 1.0

    def update(self, p_t):
        s_pred = self.s_hat
        P_pred = self.P + self.Q
        K = P_pred / (P_pred + self.R)
        self.s_hat = s_pred + K * (p_t - s_pred)
        self.P = (1 - K) * P_pred
        return self.s_hat

    def smooth_sequence(self, probs):
        self.reset()
        return [self.update(float(p)) for p in probs]

Overwriting kalman.py


In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelectiveSSM(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # Ajustamos para que cada canal tenga su propio lambda o se expanda correctamente
        self.log_lambda = nn.Parameter(torch.rand(d_model, d_state))

        self.proj_delta = nn.Linear(d_model, d_model, bias=True)
        self.proj_B     = nn.Linear(d_model, d_state, bias=False)
        self.proj_C     = nn.Linear(d_model, d_state, bias=False)

        self.out_proj = nn.Linear(d_model * d_state, d_model)

    def forward(self, x):
        B, L, D = x.shape
        N = self.d_state

        lam   = F.softplus(self.log_lambda) # (D, N)
        delta = F.softplus(self.proj_delta(x)).unsqueeze(-1) # (B, L, D, 1)
        Bt    = self.proj_B(x).unsqueeze(2)  # (B, L, 1, N)
        Ct    = self.proj_C(x).unsqueeze(2)  # (B, L, 1, N)

        # Discretización simplificada
        A_d = torch.exp(-delta * lam) # (B, L, D, N)
        B_d = delta * Bt # (B, L, D, N) - Simplificado para evitar división por cero compleja

        # Inicializamos h con la dimensión de d_model también: (B, D, N)
        h = torch.zeros(B, D, N, device=x.device)

        ys = []
        for t in range(L):
            h  = A_d[:, t, :, :] * h + B_d[:, t, :, :]
            yt = h * Ct[:, t, :, :] # Element-wise product
            ys.append(yt.view(B, 1, -1)) # Aplanamos D*N para la proyección de salida

        out = torch.cat(ys, dim=1)
        out = self.out_proj(out)
        return out

class TemporalBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, kernel_size: int = 3):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.conv = nn.Conv1d(
            d_model, d_model, kernel_size,
            padding=kernel_size // 2,
            groups=d_model
        )
        self.ssm  = SelectiveSSM(d_model, d_state)
        self.gate = nn.Linear(d_model, d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        # Conv1d espera (B, C, L)
        xc = self.conv(x.transpose(1, 2)).transpose(1, 2)
        xs = self.ssm(xc)
        g  = torch.sigmoid(self.gate(x))
        x  = self.proj(xs * g)
        return x + residual

class SpectralBlock(nn.Module):
    def __init__(self, d_model: int, seq_len: int, K: int = 32):
        super().__init__()
        self.K = K
        self.seq_len = seq_len
        self.filter_real = nn.Parameter(torch.randn(d_model, K) * 0.02)
        self.filter_imag = nn.Parameter(torch.randn(d_model, K) * 0.02)

    def forward(self, x):
        B, L, D = x.shape
        X = torch.fft.rfft(x, dim=1)
        # Promedio de magnitud para encontrar frecuencias dominantes
        mag = X.abs().mean(dim=(0, 2))
        k_actual = min(self.K, X.shape[1])
        topk_idx = mag.topk(k_actual).indices

        X_k = X[:, topk_idx, :]
        filt = torch.complex(self.filter_real[:, :k_actual], self.filter_imag[:, :k_actual])

        # Aplicar filtro
        X_k = X_k * filt.T.unsqueeze(0)

        X_out = torch.zeros(B, X.shape[1], D, dtype=torch.cfloat, device=x.device)
        X_out[:, topk_idx, :] = X_k
        out = torch.fft.irfft(X_out, n=L, dim=1)
        return out

class Fusion(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(d_model) * 0.5)
        self.beta  = nn.Parameter(torch.ones(d_model) * 0.5)

    def forward(self, x_temp, x_spec):
        return self.alpha * x_temp + self.beta * x_spec

class AnomalyDetector(nn.Module):
    def __init__(self, input_dim, d_model=64, d_state=16,
                 seq_len=32, K=16, n_layers=2, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.temporal_blocks = nn.ModuleList([TemporalBlock(d_model, d_state) for _ in range(n_layers)])
        self.spectral = SpectralBlock(d_model, seq_len, K)
        self.fusion   = Fusion(d_model)
        self.norm_out = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(d_model // 2, n_classes)
        )

    def forward(self, x):
        x = self.input_proj(x)
        xt = x
        for block in self.temporal_blocks:
            xt = block(xt)
        xs = self.spectral(x)
        z  = self.fusion(xt, xs)
        z  = self.norm_out(z)
        # Clasificamos usando el último paso de tiempo
        return self.head(z[:, -1, :])

Overwriting model.py


In [ ]:
%%writefile data.py
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from imblearn.combine import SMOTETomek
from torch.utils.data import Dataset, DataLoader
import torch

class UNSWNB15Dataset(Dataset):
    def __init__(self, X, y, seq_len=32):
        self.X, self.y = [], []
        for i in range(len(X) - seq_len):
            self.X.append(X[i:i+seq_len])
            self.y.append(y[i+seq_len-1])
        self.X = np.array(self.X, dtype=np.float32)
        self.y = np.array(self.y, dtype=np.int64)

    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])

def load_data(csv_paths, seq_len=32):
    dfs = [pd.read_csv(p, low_memory=False, encoding='utf-8-sig') for p in csv_paths]
    df  = pd.concat(dfs, ignore_index=True)

    label_col = 'label' if 'label' in df.columns else 'Label'
    y = df[label_col].values.astype(int)

    drop_cols = [label_col, 'attack_cat', 'id']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    for c in df.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df[c] = le.fit_transform(df[c].astype(str))

    df = df.fillna(0)
    X  = df.values.astype(np.float32)

    n       = len(X)
    n_test  = int(n * 0.15)
    n_val   = int(n * 0.15)
    n_train = n - n_test - n_val

    X_train, y_train = X[:n_train], y[:n_train]
    X_val,   y_val   = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
    X_test,  y_test  = X[n_train+n_val:], y[n_train+n_val:]

    scaler  = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    sm = SMOTETomek(random_state=42)
    X_train, y_train = sm.fit_resample(X_train, y_train)

    train_ds = UNSWNB15Dataset(X_train, y_train, seq_len)
    val_ds   = UNSWNB15Dataset(X_val,   y_val,   seq_len)
    test_ds  = UNSWNB15Dataset(X_test,  y_test,  seq_len)

    return (DataLoader(train_ds, batch_size=32, shuffle=True),
            DataLoader(val_ds,   batch_size=64, shuffle=False),
            DataLoader(test_ds,  batch_size=64, shuffle=False),
            X_train.shape[1])

Overwriting data.py


In [ ]:
%%writefile train.py
import torch
import torch.nn as nn
from evaluate import evaluate

def train(model, train_loader, val_loader,
          epochs=10, lr=1e-3, device='cuda', save_path='checkpoint.pt'):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_f1   = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0, 0, 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * len(y_batch)
            correct    += (model(X_batch).argmax(1) == y_batch).sum().item()
            total      += len(y_batch)

        f1, acc, *_ = evaluate(model, val_loader, device)
        print(f"Epoch {epoch}/{epochs} | "
              f"Loss: {total_loss/total:.4f} | "
              f"Val Acc: {acc:.4f} | Val F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), save_path)
            print(f"  ✓ Checkpoint guardado (F1={best_f1:.4f})")

Overwriting train.py


In [ ]:
%%writefile evaluate.py
import torch
import numpy as np
from sklearn.metrics import (accuracy_score, recall_score,
                             f1_score, mean_absolute_error, mean_squared_error)

def evaluate(model, loader, device='cuda'):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            probs = torch.softmax(model(X_batch.to(device)), dim=-1)[:,1].cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend((probs >= 0.5).astype(int))
            all_labels.extend(y_batch.numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    return (f1_score(y_true, y_pred, average='macro', zero_division=0),
            accuracy_score(y_true, y_pred),
            recall_score(y_true, y_pred, average='macro', zero_division=0),
            mean_absolute_error(y_true, y_prob),
            mean_squared_error(y_true, y_prob),
            y_prob, y_true)

Overwriting evaluate.py


In [ ]:
 import torch
from model import AnomalyDetector
from data import load_data
from train import train
from evaluate import evaluate
from kalman import KalmanSmoother
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando: {device}")

# ✅ Solo estos dos archivos
archivos = [
    './data/UNSW_NB15_training-set.csv',
    './data/UNSW_NB15_testing-set.csv'
]

train_loader, val_loader, test_loader, n_features = load_data(archivos, seq_len=32)
print(f"✅ Datos cargados | Features: {n_features}")

model = AnomalyDetector(
    input_dim=n_features, d_model=64, d_state=16,
    seq_len=32, K=16, n_layers=2
).to(device)

train(model, train_loader, val_loader, epochs=10, device=device)

model.load_state_dict(torch.load('checkpoint.pt'))

f1, acc, rec, mae, mse, probs, labels = evaluate(model, test_loader, device)
print(f"\n=== BASELINE ===")
print(f"Accuracy: {acc:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f}")

configs = [(1e-4, 1e-1), (1e-3, 1e-1), (1e-2, 1e-2)]
for sw, sv in configs:
    kal = KalmanSmoother(sw, sv)
    smoothed = np.array(kal.smooth_sequence(probs))
    yp = (smoothed >= 0.5).astype(int)
    print(f"\n=== Kalman σ²w={sw} σ²v={sv} ===")
    print(f"Accuracy: {accuracy_score(labels,yp):.4f} | "
          f"Recall: {recall_score(labels,yp,average='macro',zero_division=0):.4f} | "
          f"F1: {f1_score(labels,yp,average='macro',zero_division=0):.4f} | "
          f"MAE: {mean_absolute_error(labels,smoothed):.4f} | "
          f"MSE: {mean_squared_error(labels,smoothed):.4f}")

Usando: cuda
✅ Datos cargados | Features: 42
Epoch 1/10 | Loss: 0.0839 | Val Acc: 0.9467 | Val F1: 0.8501
  ✓ Checkpoint guardado (F1=0.8501)
Epoch 2/10 | Loss: 0.0432 | Val Acc: 0.9339 | Val F1: 0.8303
Epoch 3/10 | Loss: 0.0316 | Val Acc: 0.9382 | Val F1: 0.8342
Epoch 4/10 | Loss: 0.0242 | Val Acc: 0.9420 | Val F1: 0.8450
Epoch 5/10 | Loss: 0.0199 | Val Acc: 0.9433 | Val F1: 0.8432
Epoch 6/10 | Loss: 0.0164 | Val Acc: 0.9324 | Val F1: 0.8196
Epoch 7/10 | Loss: 0.0140 | Val Acc: 0.9501 | Val F1: 0.8569
  ✓ Checkpoint guardado (F1=0.8569)
Epoch 8/10 | Loss: 0.0123 | Val Acc: 0.9368 | Val F1: 0.8298
Epoch 9/10 | Loss: 0.0107 | Val Acc: 0.9337 | Val F1: 0.8219
Epoch 10/10 | Loss: 0.0094 | Val Acc: 0.9476 | Val F1: 0.8535

=== BASELINE ===
Accuracy: 0.9999 | Recall: 0.4999 | F1: 0.5000 | MAE: 0.0014 | MSE: 0.0003

=== Kalman σ²w=0.0001 σ²v=0.1 ===
Accuracy: 1.0000 | Recall: 1.0000 | F1: 1.0000 | MAE: 0.0014 | MSE: 0.0000

=== Kalman σ²w=0.001 σ²v=0.1 ===
Accuracy: 1.0000 | Recall: 1.0000 |

In [ ]:
import os

# Ver qué archivos hay
archivos = os.listdir('./data')
print("Archivos descargados:")
for f in archivos:
    print(f" -", f)


In [ ]:
import chardet

# Detectar encoding de cada archivo
for f in archivos:
    if f.endswith('.csv'):
        ruta = f'./data/{f}'
        with open(ruta, 'rb') as file:
            muestra = file.read(100000)  # lee los primeros 100kb
            resultado = chardet.detect(muestra)
            print(f"{f} → encoding detectado: {resultado['encoding']} (confianza: {resultado['confidence']:.0%})")